In [3]:
pip install python-calamine

   ---------------------------------------- 0.0/777.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/777.6 kB ? eta -:--:--
   ------------- -------------------------- 262.1/777.6 kB ? eta -:--:--
   ------------- -------------------------- 262.1/777.6 kB ? eta -:--:--
   ------------- -------------------------- 262.1/777.6 kB ? eta -:--:--
   ------------------------- ------------ 524.3/777.6 kB 479.2 kB/s eta 0:00:01
   ------------------------- ------------ 524.3/777.6 kB 479.2 kB/s eta 0:00:01
   -------------------------------------- 777.6/777.6 kB 499.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
#BISCUIT

import pandas as pd
import os
import glob
import numpy as np

# 1. Pengaturan Awal
folder_path = 'D:/01. Project/Top 10/Biscuit/2026 06 - 2026 08'
folder_hasil = 'D:/01. Project/Top 10'
nama_kategori = 'biscuit'  # <--- Variabel baru untuk nama file akhir
path_mapping = os.path.join(folder_hasil, 'map_market_biscuit.xlsx') 

daftar_file = glob.glob(os.path.join(folder_path, '**', '*.xls*'), recursive=True)

if not daftar_file:
    print(f"Tidak ada file Excel yang ditemukan di folder: {folder_path}")
else:
    print(f"Ditemukan {len(daftar_file)} file Excel. Memulai proses gabung...\n")

kumpulan_df = []
info_kolom_per_file = {}

# 2. Proses Pemindaian dan Pembacaan
for file in daftar_file:
    nama_file = os.path.basename(file)
    try:
        df_temp = pd.read_excel(file, sheet_name=0, header=None, nrows=15, engine='calamine', keep_default_na=False)
        baris_header = -1
        for index, row in df_temp.iterrows():
            baris_teks = [str(val).strip() for val in row.values]
            if any("markets" in teks.lower() for teks in baris_teks) or any("periods" in teks.lower() for teks in baris_teks):
                baris_header = index
                break
                
        if baris_header == -1:
            continue
            
        df_asli = pd.read_excel(file, sheet_name=0, header=baris_header, engine='calamine', keep_default_na=False)
        df_asli = df_asli.loc[:, ~df_asli.columns.str.contains('^Unnamed')]
        
        # --- FITUR 1: POTONG BARIS KOSONG & FOOTER ---
        kolom_market = [col for col in df_asli.columns if 'market' in str(col).lower()]
        if kolom_market:
            nama_kol_market = kolom_market[0]
            df_asli = df_asli[df_asli[nama_kol_market].astype(str).str.strip() != '']
            
            kata_terlarang = ['exported:', 'dataset:', 'copyright', 'terms & conditions']
            mask_footer = df_asli[nama_kol_market].astype(str).str.lower().apply(
                lambda x: any(kata in x for kata in kata_terlarang)
            )
            df_asli = df_asli[~mask_footer]
        # ---------------------------------------------
        
        df_asli['Sumber_File'] = nama_file
        info_kolom_per_file[nama_file] = set(df_asli.columns)
        kumpulan_df.append(df_asli)
        print(f"Berhasil membaca '{nama_file}' (Header di baris ke-{baris_header + 1})")
    except Exception as e:
        print(f"❌ GAGAL MEMBACA '{nama_file}'. Error: {e}")

# 3. Proses Penggabungan
if kumpulan_df:
    df_gabungan = pd.concat(kumpulan_df, ignore_index=True)
    
    # ---> FITUR BARU: HAPUS KOLOM SPESIFIK JIKA ADA <---
    kolom_hapus = ["Sales (PCS) (000)", "Sales (GR) (000)"]
    df_gabungan = df_gabungan.drop(columns=kolom_hapus, errors='ignore')
    print(f"🗑️ Menghapus kolom spesifik (jika ada): {', '.join(kolom_hapus)}")
    # ---------------------------------------------------

    # ---> NETRALISIR PERBEDAAN KOLOM (Ubah NaN jadi Teks Kosong) <---
    for col in df_gabungan.columns:
        if df_gabungan[col].dtype == 'object':
            df_gabungan[col] = df_gabungan[col].fillna('')
    # -------------------------------------------------------------------

    kolom_market_gabungan = [col for col in df_gabungan.columns if 'market' in str(col).lower()][0]
    
    # --- FITUR 3 (DIPINDAH KE ATAS): MAPPING MARKET BARU & UNIVERSE ---
    print("\n🔄 Memproses Mapping Market...")
    universe_dict = {} 
    try:
        df_mapping = pd.read_excel(path_mapping)
        # Pengaman: pastikan tidak ada market_lama ganda di excel mapping yang bikin data berlipat
        df_mapping = df_mapping.drop_duplicates(subset=['market_lama'])
        
        for _, row in df_mapping.iterrows():
            m_baru = str(row['market_baru']).strip() if pd.notna(row['market_baru']) else str(row['market_lama']).strip()
            univ = pd.to_numeric(row['universe'], errors='coerce')
            if pd.notna(univ): universe_dict[m_baru] = univ

        df_gabungan[kolom_market_gabungan] = df_gabungan[kolom_market_gabungan].astype(str).str.strip()
        df_mapping['market_lama'] = df_mapping['market_lama'].astype(str).str.strip()
        
        df_gabungan = pd.merge(df_gabungan, df_mapping, left_on=kolom_market_gabungan, right_on='market_lama', how='left')
        df_gabungan[kolom_market_gabungan] = df_gabungan['market_baru'].fillna(df_gabungan[kolom_market_gabungan])
        df_gabungan = df_gabungan.drop(columns=['market_lama', 'market_baru'])
        print("✅ Mapping selesai. Nama market telah diperbarui.")
    except Exception as e:
        print(f"⚠️ GAGAL MELAKUKAN MAPPING. Error: {e}")
    # -----------------------------------------------

    # --- FITUR 2 (DIPINDAH KE SINI): DETEKSI & HAPUS DUPLIKAT ---
    print("\n🔍 Memeriksa dan Menghapus Duplikat...")
    kolom_data = [col for col in df_gabungan.columns if col != 'Sumber_File']
    duplikat_mask = df_gabungan.duplicated(subset=kolom_data, keep=False)
    if duplikat_mask.any():
        df_duplikat = df_gabungan[duplikat_mask]
        rekap_duplikat = df_duplikat.groupby(['Sumber_File', kolom_market_gabungan], dropna=False).size()
        
        print("⚠️ LAPORAN DATA DUPLIKAT ⚠️")
        for (nama_f, market), jml in rekap_duplikat.items():
            print(f"   - File: '{nama_f}' | Market: '{market}' -> {jml} baris terlibat")
            
        jumlah_sebelum = len(df_gabungan)
        df_gabungan = df_gabungan.drop_duplicates(subset=kolom_data, ignore_index=True)
        jumlah_sesudah = len(df_gabungan)
        print(f"🗑️ Membuang Duplikat: Berhasil menghapus {jumlah_sebelum - jumlah_sesudah} baris berlebih.")
    else:
        print("✅ Tidak ada data duplikat yang ditemukan antar file. Data sangat bersih.")
    # -----------------------------------------

    # PERSIAPAN DIMENSI DAN FAKTA UNTUK CUSTOM MARKET
    kolom_sum = ['Sales (GR)', 'Sales (PCS)', 'Sales Value (MM)', 'Stocks Forward (GR)', 
                 'Stocks (GR)', 'Purchase (GR)', 'Stocks Forward (PCS)', 'Purchase (PCS)', 'Stocks (PCS)']
    kolom_wgt = ['Numeric Distribution Handling (C)', 'Numeric Out of Stock (C)', 
                 'Wghtd Dist Handling (C)', 'Numeric Distribution Purchasing (C)', 
                 'Numeric Distribution Selling (C)', 'Numeric Distribution Stocking (C)']
    dimensi_mentah = ['Periods', 'BISCUIT', 'SEASONALITY', 'MANUFACTURER', 'SEGMENT', 
                      'SUBSEGMENT', 'BRAND', 'SUBVARIANT', 'FLAVOUR', 'PRICE SEGMENT', 
                      'SIZE', 'PACKAGE', 'PACKSIZE', 'ITEM']

    valid_sum = [c for c in kolom_sum if c in df_gabungan.columns]
    valid_wgt = [c for c in kolom_wgt if c in df_gabungan.columns]
    valid_dim = [c for c in dimensi_mentah if c in df_gabungan.columns]
    
    df_gabungan['universe'] = pd.to_numeric(df_gabungan['universe'], errors='coerce')
    for c in valid_sum + valid_wgt:
        df_gabungan[c] = pd.to_numeric(df_gabungan[c], errors='coerce')
        
    temp_wgt_cols = []
    for c in valid_wgt:
        temp_c = f"{c}_temp_wgt"
        df_gabungan[temp_c] = df_gabungan[c] * df_gabungan['universe']
        temp_wgt_cols.append(temp_c)

    # --- FITUR 4: CUSTOM MARKET TAMBAH ---
    print("\n🏗️ Membangun Custom Market (Penjumlahan)...")
    custom_markets_tambah = {
        'DPL TOTAL': ['Jakarta Greater', 'West Java', 'Central Java + DIY', 'East Java'],
        'DPL UMUM': ['JKT Greater GT', 'West Java GT', 'Central Java + DIY GT', 'East Java GT'],
        'DPL P90': ['JKT Greater MT', 'West Java MT', 'Central Java + DIY MT', 'East Java MT'],
        'LPL TOTAL': ['Sumatra', 'Other Islands'],
        'LPL UMUM': ['Sumatra General Trade', 'Other Islands General Trade'],
        'LPL P90': ['Sumatra MT', 'Other Islands MT'],
        'DPB TOTAL': ['Jakarta Greater', 'West Java'],
        'DPB UMUM': ['JKT Greater GT', 'West Java GT'],
        'DPB P90': ['JKT Greater MT', 'West Java MT'],
        'DPT TOTAL': ['Central Java + DIY', 'East Java'],
        'DPT UMUM': ['Central Java + DIY GT', 'East Java GT'],
        'DPT P90': ['Central Java + DIY MT', 'East Java MT'],
        '16 KOTA TOTAL': ['Jakarta Greater', 'Bandung Greater', 'Semarang Greater', 'Yogyakarta Greater', 'Surabaya Greater', 'Greater Malang', 'Denpasar', 'North Sumatra Medan', 'Pekanbaru', 'Batam', 'Palembang', 'Bandar Lampung', 'Banjarmasin', 'Pontianak', 'Manado', 'Sulawesi Makassar Greater']
    }
    
    list_df_tambah = []
    for new_market, komponen in custom_markets_tambah.items():
        df_filter = df_gabungan[df_gabungan[kolom_market_gabungan].isin(komponen)].copy()
        if df_filter.empty: continue
            
        agg_dict = {'universe': 'sum'}
        for c in valid_sum + temp_wgt_cols: agg_dict[c] = 'sum'
        
        df_grouped = df_filter.groupby(valid_dim, dropna=False).agg(agg_dict).reset_index()
        for c, t in zip(valid_wgt, temp_wgt_cols):
            df_grouped[c] = np.where(df_grouped['universe'] > 0, df_grouped[t] / df_grouped['universe'], np.nan)
            df_grouped = df_grouped.drop(columns=[t])
            
        df_grouped[kolom_market_gabungan] = new_market
        list_df_tambah.append(df_grouped)

    # --- FITUR 5: CUSTOM MARKET KURANG ---
    print("✂️ Membangun Custom Market (Pengurangan)...")
    custom_markets_kurang = {
        'North Sumatra MT': ['North Sumatra', 'North Sumatra GT'],
        'Central Sumatra MT': ['Central Sumatra', 'Central Sumatra GT'],
        'Southern Sumatra MT': ['Southern Sumatra', 'Southern Sumatra GT'],
        'West Java MT Hyper/Super': ['West Java MT', 'West Java MT Minimarket'],
        'Central Java + DIY MT Hyper/Super': ['Central Java + DIY MT', 'Central Java + DIY MT Minimarket'],
        'East Java MT Hyper/Super': ['East Java MT', 'East Java MT Minimarket']
    }

    df_sumatra_mt = df_gabungan[df_gabungan[kolom_market_gabungan] == 'Sumatra MT'].copy()
    if not df_sumatra_mt.empty:
        for col in valid_dim: df_sumatra_mt[col] = df_sumatra_mt[col].fillna('__NAN__') 
        df_sumatra_wgt = df_sumatra_mt.groupby(valid_dim)[valid_wgt].first().reset_index()
    else:
        df_sumatra_wgt = pd.DataFrame(columns=valid_dim + valid_wgt)

    list_df_kurang = []
    for new_market, (m1, m2) in custom_markets_kurang.items():
        df_m1 = df_gabungan[df_gabungan[kolom_market_gabungan] == m1].copy()
        df_m2 = df_gabungan[df_gabungan[kolom_market_gabungan] == m2].copy()
        
        if df_m1.empty and df_m2.empty: continue
            
        df_m1['multiplier'] = 1
        df_m2['multiplier'] = -1
        df_calc = pd.concat([df_m1, df_m2], ignore_index=True)
        
        for c in valid_sum + temp_wgt_cols:
            df_calc[c] = df_calc[c] * df_calc['multiplier']
            
        agg_dict = {}
        for c in valid_sum + temp_wgt_cols: agg_dict[c] = 'sum'
            
        df_grouped = df_calc.groupby(valid_dim, dropna=False).agg(agg_dict).reset_index()
        df_grouped['universe'] = universe_dict.get(new_market, np.nan)
        
        for c in valid_sum:
            df_grouped[c] = df_grouped[c].clip(lower=0)
            
        if new_market in ['North Sumatra MT', 'Central Sumatra MT', 'Southern Sumatra MT'] and not df_sumatra_wgt.empty:
            for col in valid_dim: df_grouped[col] = df_grouped[col].fillna('__NAN__')
            df_grouped = pd.merge(df_grouped, df_sumatra_wgt, on=valid_dim, how='left')
            for col in valid_dim: df_grouped[col] = df_grouped[col].replace('__NAN__', np.nan)
        else:
            for c, t in zip(valid_wgt, temp_wgt_cols):
                df_grouped[c] = np.where(df_grouped['universe'] > 0, df_grouped[t] / df_grouped['universe'], np.nan)
                df_grouped[c] = df_grouped[c].clip(lower=0, upper=100)
                
        df_grouped = df_grouped.drop(columns=temp_wgt_cols, errors='ignore')
        df_grouped[kolom_market_gabungan] = new_market
        list_df_kurang.append(df_grouped)

    df_gabungan = df_gabungan.drop(columns=temp_wgt_cols)

    if list_df_tambah or list_df_kurang:
        df_semua_baru = pd.concat(list_df_tambah + list_df_kurang, ignore_index=True)
        df_gabungan = pd.concat([df_gabungan, df_semua_baru], ignore_index=True)
        print(f"✅ Sukses menambahkan Custom Market!")

    # --- FITUR 6: EKSTRAK TAHUN DAN BULAN ---
    print("\n📅 Mengekstrak Tahun dan Bulan dari kolom Periods...")
    if 'Periods' in df_gabungan.columns:
        ekstrak_waktu = df_gabungan['Periods'].astype(str).str.extract(r'\d{2}/(\d{2})/(\d{2,4})')
        df_gabungan['Bulan'] = ekstrak_waktu[0]
        df_gabungan['Tahun'] = ekstrak_waktu[1].str.replace(r'^(\d{2})$', r'20\1', regex=True)
        df_gabungan = df_gabungan.drop(columns=['Periods'])
        print("✅ Kolom 'Bulan' dan 'Tahun' berhasil dibuat!")
    else:
        print("   ⚠️ Kolom 'Periods' tidak ditemukan, ekstrak waktu dilewati.")
    # ----------------------------------------

    # --- PEMBERSIHAN FINAL & EXPORT PARQUET ---
    if 'Sumber_File' in df_gabungan.columns:
        df_gabungan = df_gabungan.drop(columns=['Sumber_File'])
    
    for col in df_gabungan.columns:
        if df_gabungan[col].dtype == 'object' or df_gabungan[col].dtype.name == 'category':
            # Sekaligus membersihkan spasi ekstra di awal/akhir kata yang sering bikin data ganda bayangan
            df_gabungan[col] = df_gabungan[col].fillna('').astype(str).str.strip()
            
    # SAPU JAGAT TERAKHIR (Untuk menangkap duplikat akibat proses ekstrak waktu & pembersihan spasi)
    jml_sblm = len(df_gabungan)
    df_gabungan = df_gabungan.drop_duplicates(ignore_index=True)
    jml_ssdh = len(df_gabungan)
    if jml_sblm != jml_ssdh:
        print(f"🧹 SAPU JAGAT: Menghapus {jml_sblm - jml_ssdh} duplikat akhir akibat spasi/ekstrak tanggal.")
            
    print("\n=======================================================")
    print("Memecah dan menyimpan file Parquet berdasarkan Bulan & Tahun...")
    
    if 'Bulan' in df_gabungan.columns and 'Tahun' in df_gabungan.columns:
        for (tahun, bulan), df_part in df_gabungan.groupby(['Tahun', 'Bulan']):
            if pd.isna(tahun) or pd.isna(bulan) or tahun == '' or bulan == '':
                nama_file_parquet = f"unknown_{nama_kategori}.parquet"
            else:
                nama_file_parquet = f"{tahun}{bulan}_{nama_kategori}.parquet"
                
            output_parquet = os.path.join(folder_hasil, nama_file_parquet)
            df_part.to_parquet(output_parquet, index=False)
            print(f" 💾 Berhasil menyimpan: {nama_file_parquet} ({len(df_part)} baris)")
    else:
        nama_file_parquet = f"gabungan_{nama_kategori}.parquet"
        output_parquet = os.path.join(folder_hasil, nama_file_parquet)
        df_gabungan.to_parquet(output_parquet, index=False)
        print(f" 💾 Berhasil menyimpan: {nama_file_parquet} ({len(df_gabungan)} baris)")

    print("=======================================================\n")
    print(f"SELESAI KESELURUHAN! Total baris diproses: {len(df_gabungan)}")

    # =======================================================
    # 4. LAPORAN KELENGKAPAN KOLOM
    # =======================================================
    print("\n=======================================================")
    print("LAPORAN KELENGKAPAN KOLOM ANTAR FILE:")
    
    semua_kolom_asli = set()
    for kolom_file in info_kolom_per_file.values():
        semua_kolom_asli.update(kolom_file)
        
    semua_kolom_asli = semua_kolom_asli - {'Sumber_File'}
    
    ada_yang_kurang = False
    for nama_file, kolom_file in info_kolom_per_file.items():
        kolom_file_bersih = kolom_file - {'Sumber_File'}
        kolom_kurang = semua_kolom_asli - kolom_file_bersih 
        
        if kolom_kurang:
            ada_yang_kurang = True
            print(f" ⚠️ File '{nama_file}' kekurangan {len(kolom_kurang)} kolom: {', '.join(kolom_kurang)}")
            
    if not ada_yang_kurang:
        print(" ✅ Mantap! Semua file memiliki jumlah & nama kolom asli yang sama persis.")
    print("=======================================================\n")

Ditemukan 55 file Excel. Memulai proses gabung...

Berhasil membaca 'All Item Biscuit F_Text File a (1).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File a (2).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File a (3).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File b (1).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File b (2).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File b (3).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File C (1).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File C (2).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File C (3).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File D (1).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File D (2).xlsx' (Header di baris ke-9)
Berhasil membaca 'All Item Biscuit F_Text File 

C:\Users\921006307\AppData\Local\Temp\ipykernel_10420\2481388125.py:75: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_gabungan[col] = df_gabungan[col].fillna('')



🔄 Memproses Mapping Market...
✅ Mapping selesai. Nama market telah diperbarui.

🔍 Memeriksa dan Menghapus Duplikat...
⚠️ LAPORAN DATA DUPLIKAT ⚠️
   - File: 'All Item Biscuit F_Text File C (1).xlsx' | Market: 'Sumatra GT Urban' -> 96 baris terlibat
   - File: 'All Item Biscuit F_Text File C (1).xlsx' | Market: 'Sumatra Urban' -> 466 baris terlibat
   - File: 'All Item Biscuit F_Text File C (2).xlsx' | Market: 'Sumatra GT Urban' -> 85 baris terlibat
   - File: 'All Item Biscuit F_Text File C (2).xlsx' | Market: 'Sumatra Urban' -> 405 baris terlibat
   - File: 'All Item Biscuit F_Text File C (3).xlsx' | Market: 'Sumatra GT Urban' -> 76 baris terlibat
   - File: 'All Item Biscuit F_Text File C (3).xlsx' | Market: 'Sumatra Urban' -> 373 baris terlibat
   - File: 'All Item Biscuit F_Text File D (1).xlsx' | Market: 'Indonesia MT Independent' -> 1760 baris terlibat
   - File: 'All Item Biscuit F_Text File D (1).xlsx' | Market: 'Sumatra' -> 469 baris terlibat
   - File: 'All Item Biscuit F_Te

In [4]:
import duckdb
import pandas as pd
path = 'G:/Shared drives/00 CBDI/01 Reporting/DB/Textfile/Biscuit/*.parquet'
# path = 'Biscuit.parquet'

# query = """
#     SELECT *
#     FROM '""" + path + """'
#     WHERE 
#     MANUFACTURER = '' 
#     AND PERIODS = 'Apr 26 - w/e 22/04/26'
#     AND SEGMENT = '' 
#     AND SUBSEGMENT = '' 
#     AND BRAND = '' 
#     AND SUBVARIANT = '' 
#     AND FLAVOUR = '' 
#     --AND PRICE SEGMENT = '' 
#     AND SIZE = '' 
#     --AND PACKAGE = '' 
#     --AND PACKSIZE = '' 
#     --AND ITEM = '' 

# """

query = """
    SELECT *
    FROM '""" + path + """'
    WHERE SUBVARIANT = '' AND TAHUN = 2026 AND MANUFACTURER = '' AND MARKETS = 'Indonesia'
"""

# 1. Eksekusi kueri dan ubah menjadi DataFrame Pandas
df_hasil = duckdb.query(query).df()

df_hasil.to_csv('biscuit_2026.csv')
# 2. Tampilkan hasilnya
# print(df_hasil.T)

In [3]:
import duckdb
import pandas as pd
path = 'G:/Shared drives/00 CBDI/01 Reporting/DB/Textfile/Candies/*.parquet'
query = """
    SELECT *
    FROM '""" + path + """'
    WHERE TAHUN = 2026 AND BULAN = 1 AND (MARKET LIKE '%DPL%' OR MARKET LIKE '%LPL%') 
"""

# 1. Eksekusi kueri dan ubah menjadi DataFrame Pandas
df_hasil = duckdb.query(query).df()

df_hasil.to_csv('candy_DPL_LPL.csv')
# 2. Tampilkan hasilnya
# print(df_hasil.T)

In [60]:
import pandas as pd
import os
import glob
import gc

# ==========================================
# 1. PENGATURAN LOKASI FILE
# ==========================================
folder_txt = 'D:/01. Project/Top 10/Candy Old'
file_header_csv = 'D:/01. Project/Top 10/Candy Old/nama_kolom_old.csv' 
file_mapping_market = 'D:/01. Project/Top 10/Candy Old/market_candy_old.csv'

pemisah_txt = ',' 
# ==========================================

print("⏳ Memulai proses Candy Old (Mode 1 per 1, Fix DuckDB & Weighted ND/WD)...\n")

try:
    # 2. BACA FILE MAPPING & UNIVERSE
    df_header = pd.read_csv(file_header_csv, header=None, keep_default_na=False)
    daftar_kolom = df_header[0].astype(str).str.strip().tolist()
    print(f"✅ Berhasil membaca {len(daftar_kolom)} nama kolom.")
    
    df_map_market = pd.read_csv(file_mapping_market, keep_default_na=False, dtype=str)
    
    # Kamus ganti nama market
    kamus_market = dict(zip(df_map_market['market_lama'].str.strip(), df_map_market['market_baru'].str.strip()))
    
    # Kamus Universe (Cari kolom yang mengandung kata 'universe')
    kolom_univ = [c for c in df_map_market.columns if 'universe' in str(c).lower()]
    if kolom_univ:
        # Kita pasangkan Universe dengan market_baru karena data akan digabung setelah ganti nama
        kamus_universe = dict(zip(df_map_market['market_baru'].str.strip(), pd.to_numeric(df_map_market[kolom_univ[0]], errors='coerce').fillna(0)))
        print(f"✅ Berhasil memuat data UNIVERSE dari kolom '{kolom_univ[0]}' untuk perhitungan bobot ND.")
    else:
        kamus_universe = {}
        print("⚠️ PERINGATAN: Kolom 'Universe' tidak ditemukan di market_candy_old.csv!")

    daftar_file_txt = glob.glob(os.path.join(folder_txt, '*.txt'))
    
    if not daftar_file_txt:
        print(f"❌ Tidak ada file .txt di: {folder_txt}")
    else:
        target_dimensi = [
            'Tahun', 'Bulan', 'Market', 'INDUSTRI', 'SEASONALITY', 'SEGMENT', 
            'SUBSEGMENT', 'PRICE_SEGMENT', 'MANUFACTURER', 'BRAND', 'SUBBRAND', 
            'KELP_ITEM', 'VARIANT', 'FLAVOUR', 'PACKTYPE', 'SIZE_ITEM', 
            'FUNCTION_ITEM', 'PACKSIZE', 'ITEM', 'BARCODE', 'FORMAT_PRODUCT', 
            'SUBFORMAT', 'OWNERSHIP', 'LEVEL_DATA'
        ]
        
        # 3. LOOPING PROSES 1 PER 1 FILE
        for file in daftar_file_txt:
            nama_file = os.path.basename(file)
            print(f"\n⚙️ Memproses: {nama_file}")
            
            try:
                # --- A. BACA FILE & BERSIHKAN ---
                df = pd.read_csv(
                    file, sep=pemisah_txt, header=None, names=daftar_kolom, dtype=str, 
                    keep_default_na=False, index_col=False, usecols=range(len(daftar_kolom))
                )
                
                for col in df.columns:
                    df[col] = df[col].astype(str).str.strip()
                    
                df = df[~(df == '').all(axis=1)]
                if df.empty:
                    print("   -> Hanya berisi baris kosong, dilewati.")
                    continue

                # --- B. MAPPING MARKET ---
                if 'Market' in df.columns:
                    df['Market'] = df['Market'].apply(lambda x: kamus_market.get(x, x))
                
                # --- C. HAPUS DUPLIKAT ---
                subset_duplikat = [c for c in df.columns if str(c).lower() in [k.lower() for k in target_dimensi]]
                if subset_duplikat:
                    df.drop_duplicates(subset=subset_duplikat, keep='first', inplace=True)
                    df = df.reset_index(drop=True)
                
                # --- D. BUAT CUSTOM MARKET (LOGIKA WEIGHTED AVERAGE BERDASARKAN UNIVERSE) ---
                custom_markets_tambah = {
                    'DPL TOTAL': ['Jakarta Greater', 'West Java', 'Central Java + DIY', 'East Java'],
                    'DPL UMUM': ['JKT Greater GT', 'West Java GT', 'Central Java + DIY GT', 'East Java GT'],
                    'DPL P90': ['JKT Greater MT', 'West Java MT', 'Central Java + DIY MT', 'East Java MT'],
                    'LPL TOTAL': ['Sumatra', 'Other Islands'],
                    'LPL UMUM': ['Sumatra General Trade', 'Other Islands General Trade'],
                    'LPL P90': ['Sumatra MT', 'Other Islands MT'],
                    'DPB TOTAL': ['Jakarta Greater', 'West Java'],
                    'DPB UMUM': ['JKT Greater GT', 'West Java GT'],
                    'DPB P90': ['JKT Greater MT', 'West Java MT'],
                    'DPT TOTAL': ['Central Java + DIY', 'East Java'],
                    'DPT UMUM': ['Central Java + DIY GT', 'East Java GT'],
                    'DPT P90': ['Central Java + DIY MT', 'East Java MT'],
                    '16 KOTA TOTAL': ['Jakarta Greater', 'Bandung Greater', 'Semarang Greater', 'Yogyakarta Greater', 'Surabaya Greater', 'Greater Malang', 'Denpasar', 'North Sumatra Medan', 'Pekanbaru', 'Batam', 'Palembang', 'Bandar Lampung', 'Banjarmasin', 'Pontianak', 'Manado', 'Sulawesi Makassar Greater']
                }
                
                kolom_nilai = [col for col in df.columns if col not in target_dimensi]
                for col in kolom_nilai:
                    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                
                kolom_distribusi = ['NDSelling', 'NDInStock', 'NDOutofStock', 'NDTotalStock', 'NPD', 'WD']
                kolom_distribusi_ada = [c for c in kolom_distribusi if c in df.columns]
                
                list_custom = []
                for market_baru, komponen in custom_markets_tambah.items():
                    df_filter = df[df['Market'].isin(komponen)].copy()
                    
                    if not df_filter.empty:
                        # 1. Tarik nilai Universe untuk masing-masing area komponen
                        df_filter['Universe'] = df_filter['Market'].map(kamus_universe).fillna(0)
                        
                        # 2. Kalikan nilai ND/WD dengan Universe-nya masing-masing
                        for col in kolom_distribusi_ada:
                            df_filter[col + '_weighted'] = df_filter[col] * df_filter['Universe']
                        
                        # 3. Ubah nama Market menjadi Custom (misal 'DPL TOTAL')
                        df_filter['Market'] = market_baru
                        
                        # 4. Tentukan aturan SUM
                        cols_to_sum = [c for c in kolom_nilai if c not in kolom_distribusi_ada]
                        cols_weighted = [c + '_weighted' for c in kolom_distribusi_ada]
                        agg_dict = {c: 'sum' for c in cols_to_sum + cols_weighted + ['Universe']}
                        
                        # 5. Lakukan Penjumlahan (SUM)
                        df_agg = df_filter.groupby(target_dimensi, dropna=False, as_index=False).agg(agg_dict)
                        
                        # 6. Bagi kembali hasil perkalian dengan Total Universe untuk mendapatkan rata-rata tertimbang
                        for col in kolom_distribusi_ada:
                            col_w = col + '_weighted'
                            df_agg[col] = df_agg.apply(lambda row: (row[col_w] / row['Universe']) if row['Universe'] > 0 else 0, axis=1)
                            df_agg.drop(columns=[col_w], inplace=True)
                            
                        # Buang kolom Universe agar tidak ikut tersimpan
                        df_agg.drop(columns=['Universe'], inplace=True)
                        list_custom.append(df_agg)
                
                if list_custom:
                    df_custom = pd.concat(list_custom, ignore_index=True)
                    df = pd.concat([df, df_custom], ignore_index=True)
                    print(f"   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).")

                # --- E. TAMBAH KOLOM KOSONG & PERBAIKI TIPE DATA (FIX DUCKDB) ---
                kolom_tambahan_teks = [
                    'SEASONALITY', 'SUBSEGMENT', 'PRICE_SEGMENT', 'KELP_ITEM', 
                    'SIZE_ITEM', 'FUNCTION_ITEM', 'BARCODE', 'FORMAT_PRODUCT', 
                    'SUBFORMAT', 'OWNERSHIP', 'LEVEL_DATA'
                ]
                for col in kolom_tambahan_teks:
                    if col not in df.columns:
                        df[col] = ''
                    else:
                        df[col] = df[col].astype(str).replace('nan', '')
                
                kolom_tambahan_angka = ['NumItem', 'JlhToko', 'SkuCarried']
                for col in kolom_tambahan_angka:
                    if col not in df.columns:
                        df[col] = 0.0
                    else:
                        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

                # --- F. SIMPAN KE PARQUET ---
                df['Bulan'] = df['Bulan'].apply(lambda x: x.zfill(2) if x.isdigit() else x)
                
                grup_waktu = df.groupby(['Tahun', 'Bulan'])
                for (tahun, bulan), df_grup in grup_waktu:
                    if tahun == '' or bulan == '':
                        nama_file_parquet = 'Unidentified_Periods_candy.parquet'
                    else:
                        nama_file_parquet = f"{tahun}{bulan}_candy.parquet"
                        
                    path_simpan = os.path.join(folder_txt, nama_file_parquet)
                    df_grup.to_parquet(path_simpan, index=False)
                    print(f"   💾 Tersimpan: {nama_file_parquet} ({len(df_grup)} baris)")
                    
                del df
                if list_custom:
                    del list_custom, df_custom
                gc.collect()
                
            except Exception as e:
                print(f"   ⚠️ GAGAL memproses {nama_file}. Error: {e}")

        print("\n=======================================================")
        print(f"🎉 SELESAI SELURUHNYA!")
        print("=======================================================")

except Exception as e:
    print(f"\n❌ Terjadi kesalahan sistem: {e}")

⏳ Memulai proses Candy Old (Mode 1 per 1, Fix DuckDB & Weighted ND/WD)...

✅ Berhasil membaca 40 nama kolom.
✅ Berhasil memuat data UNIVERSE dari kolom 'universe' untuk perhitungan bobot ND.

⚙️ Memproses: 202401 Candy - All.txt
   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).
   💾 Tersimpan: 202401_candy.parquet (740212 baris)

⚙️ Memproses: 202402 Candy - All.txt
   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).
   💾 Tersimpan: 202402_candy.parquet (745370 baris)

⚙️ Memproses: 202403 Candy - All.txt
   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).
   💾 Tersimpan: 202403_candy.parquet (739595 baris)

⚙️ Memproses: 202404 Candy - All.txt
   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).
   💾 Tersimpan: 202404_candy.parquet (745783 baris)

⚙️ Memproses: 202405 Candy - All.txt
   ➕ Custom Market berhasil ditambahkan (Menggunakan Weighted Average Universe).
   💾 T

In [18]:
import pandas as pd

textfile = 'D:/01. Project/Top 10/Candy Old/candy_old.csv'

df = pd.read_csv(textfile)


C:\Users\921006307\AppData\Local\Temp\ipykernel_22232\3608272244.py:5: DtypeWarning: Columns (3,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(textfile)


In [28]:
distinct_market = df['Bulan'].unique()

print(distinct_market)
df_market = pd.DataFrame(distinct_market, columns=['Market'])
df_market.to_csv('cek_market_candy.csv')

['Indonesia' 'Indonesia MT' 'Indonesia GT' 'Indonesia MT HS'
 'Indonesia MT MM' 'Indonesia Urban' 'Indonesia Rural'
 'Indonesia GT Urban' 'Indonesia MTI' 'Indonesia MTKA' 'Indonesia H&B'
 'Jkt Gtr' 'Jkt Gtr MT' 'Jkt Gtr GT' 'Jkt Gtr MT HS' 'Jkt Gtr MT MM'
 'W.Java' 'W.Java GT' 'W.Java MT' 'W.Java MT MM' 'C.Java + DIY'
 'C.Java + DIY GT' 'C.Java + DIY MT' 'C.Java + DIY MT MM' 'E.Java'
 'E.Java GT' 'E.Java MT' 'E.Java MT MM' 'Sum' 'Sum GT' 'Sum MT' 'N.Sum'
 'N.Sum GT' 'C.Sum' 'C.Sum GT' 'S.Sum' 'S.Sum GT' 'O.Islands'
 'O.Islands GT' 'O.Islands MT' 'Yogyakarta Gtr' 'Surabaya Gtr'
 'Malang Gtr' 'Pekanbaru' 'Batam' 'Palembang' 'Bandar Lampung'
 'Makassar Gtr' 'Banjarmasin' 'Pontianak' 'Manado' 'Kal' 'Kal Urban' 'Sul'
 'Sul Urban' 'Jkt Gtr Urban' 'Jkt Gtr GT Urban' 'W.Java Urban'
 'W.Java Rural' 'W.Java GT Urban' 'C.Java Urban' 'C.Java Rural'
 'C.Java GT Urban' 'E.Java Urban' 'E.Java Rural' 'E.Java GT Urban'
 'Sum Urban' 'Sum Rural' 'Sum GT Urban' 'N.Sum Urban' 'C.Sum Urban'
 'C.Sum GT Urban

In [56]:
import pandas as pd
import os
import glob
import gc

# ==========================================
# 1. PENGATURAN LOKASI FILE
# ==========================================
folder_excel = 'D:/01. Project/Top 10/Candy New'
file_mapping_kolom = 'D:/01. Project/Top 10/Candy New/nama_kolom_new.csv'
file_mapping_market = 'D:/01. Project/Top 10/Candy New/market_candy_new.csv' # File mapping market baru
# ==========================================

print("⏳ Memulai proses penggabungan Candy New (Konsep Parquet Bulanan)...\n")

try:
    # 2. BACA FILE MAPPING KOLOM & MARKET
    # Mapping Kolom
    df_map_kolom = pd.read_csv(file_mapping_kolom, keep_default_na=False, dtype=str)
    kamus_kolom = dict(zip(df_map_kolom['nama_lama'].str.strip(), df_map_kolom['nama_baru'].str.strip()))
    print(f"✅ Berhasil memuat {len(kamus_kolom)} aturan perubahan nama kolom.")
    
    # Mapping Market
    df_map_market = pd.read_csv(file_mapping_market, keep_default_na=False, dtype=str)
    kamus_market = dict(zip(df_map_market['market_lama'].str.strip(), df_map_market['market_baru'].str.strip()))
    print(f"✅ Berhasil memuat {len(kamus_market)} aturan perubahan nama market.")

    # 3. CARI SEMUA FILE EXCEL
    daftar_file = glob.glob(os.path.join(folder_excel, '*.xls*'))
    
    if not daftar_file:
        print(f"❌ Tidak ada file Excel yang ditemukan di folder: {folder_excel}")
    else:
        print(f"📄 Ditemukan {len(daftar_file)} file Excel. Mulai memproses...\n")
        kumpulan_data = []
        
        # 4. LOOPING MEMBACA EXCEL & PEMBERSIHAN AWAL
        for file in daftar_file:
            nama_file = os.path.basename(file)
            try:
                # --- A. DETEKSI HEADER ---
                df_temp = pd.read_excel(file, header=None, nrows=15, keep_default_na=False, dtype=str, engine='calamine')
                baris_header = -1
                for index, row in df_temp.iterrows():
                    baris_teks = [str(val).strip().lower() for val in row.values]
                    if any(k.lower() in baris_teks for k in kamus_kolom.keys()) or any(x in baris_teks for x in ['markets', 'periods']):
                        baris_header = index
                        break
                        
                if baris_header == -1:
                    print(f"   [SKIP] Header tidak ditemukan di file: {nama_file}")
                    continue
                
                # --- B. BACA DATA ASLI ---
                df_asli = pd.read_excel(file, header=baris_header, dtype=str, keep_default_na=False, engine='calamine')
                baris_kotor = len(df_asli)
                
                # --- C. PEMBASMI KOLOM HANTU & RENAME KOLOM ---
                df_asli = df_asli.loc[:, ~df_asli.columns.str.contains('^Unnamed')]
                df_asli.columns = df_asli.columns.astype(str).str.strip()
                df_asli = df_asli.rename(columns=kamus_kolom)
                
                # --- D. PEMOTONG FOOTER & BARIS KOSONG ---
                kolom_cek = [col for col in df_asli.columns if 'market' in str(col).lower() or 'period' in str(col).lower()]
                if kolom_cek:
                    kol_utama = kolom_cek[0]
                    df_asli = df_asli[df_asli[kol_utama].astype(str).str.strip() != '']
                    
                    kata_terlarang = ['exported:', 'dataset:', 'copyright', 'terms & conditions']
                    mask_footer = df_asli[kol_utama].astype(str).str.lower().apply(
                        lambda x: any(kata in x for kata in kata_terlarang)
                    )
                    df_asli = df_asli[~mask_footer]

                for col in df_asli.columns:
                    df_asli[col] = df_asli[col].astype(str).str.strip()

                df_asli['Sumber_File'] = nama_file
                kumpulan_data.append(df_asli)
                
                baris_bersih = len(df_asli)
                print(f"   -> {nama_file}: Dibaca {baris_kotor} baris | Data bersih: {baris_bersih} baris.")
                
            except Exception as e:
                print(f"   ⚠️ GAGAL membaca {nama_file}. Error: {e}")

        # ========================================================
        # 5. GABUNGKAN & PROSES DATA
        # ========================================================
        if kumpulan_data:
            print("\n🔗 Menggabungkan semua data antar file...")
            df_gabungan = pd.concat(kumpulan_data, ignore_index=True)
            df_gabungan = df_gabungan.fillna('')
            
            del kumpulan_data # Sapu memori
            gc.collect()

            # --- 5.1. MAPPING MARKET BARU ---
            print("🔄 Menerapkan Mapping Market...")
            kolom_market = [col for col in df_gabungan.columns if 'market' in str(col).lower()]
            if kolom_market:
                nama_kol_market = kolom_market[0]
                # Ubah nama market menggunakan kamus. Jika tidak ada di kamus, biarkan nama aslinya
                df_gabungan[nama_kol_market] = df_gabungan[nama_kol_market].apply(lambda x: kamus_market.get(x, x))
            
            # --- 5.2. INVESTIGASI & HAPUS DUPLIKAT ---
            print("🕵️ Memeriksa Duplikat Data berdasarkan Dimensi Kunci...")
            target_dimensi = [
                'Periods', 'Market', 'INDUSTRI', 'SEGMENT', 'MANUFACTURER', 
                'BRAND', 'SUBBRAND', 'VARIANT', 'FLAVOUR', 'PACKTYPE', 
                'PACKSIZE', 'ITEM'
            ]
            
            subset_duplikat = [c for c in df_gabungan.columns if str(c).lower() in [k.lower() for k in target_dimensi]]
            
            if subset_duplikat:
                mask_duplikat = df_gabungan.duplicated(subset=subset_duplikat, keep='first')
                if mask_duplikat.any():
                    jml_duplikat = mask_duplikat.sum()
                    print(f"⚠️ DITEMUKAN {jml_duplikat} BARIS DUPLIKAT! (Dihapus)")
                    df_gabungan.drop_duplicates(subset=subset_duplikat, keep='first', inplace=True)
                    df_gabungan = df_gabungan.reset_index(drop=True)
                else:
                    print("✅ Tidak ada duplikat berdasarkan Primary Key tersebut.")

            # Hapus kolom kamera pengintai
            if 'Sumber_File' in df_gabungan.columns:
                df_gabungan.drop(columns=['Sumber_File'], inplace=True)

            # --- 5.3. EKSTRAK TAHUN & BULAN ---
            print("📅 Mengekstrak Tahun dan Bulan...")
            kolom_periods = [col for col in df_gabungan.columns if 'periods' in str(col).lower()]
            if kolom_periods:
                nama_kol_per = kolom_periods[0]
                ekstrak_waktu = df_gabungan[nama_kol_per].astype(str).str.extract(r'\d{2}/(\d{2})/(\d{2,4})')
                df_gabungan['Bulan'] = ekstrak_waktu[0].fillna('Unknown')
                df_gabungan['Tahun'] = ekstrak_waktu[1].str.replace(r'^(\d{2})$', r'20\1', regex=True).fillna('Unknown')
                
                # Hapus kolom periods asli
                df_gabungan.drop(columns=[nama_kol_per], inplace=True)

            # --- 5.4. TAMBAH KOLOM KOSONG & PERBAIKAN TIPE DATA (FIX DUCKDB) ---
            print("➕ Menambahkan kolom kosong dan merapikan tipe data...")
            
            # A. Kolom yang bersifat TEKS (Dimensi) -> Diisi string kosong ''
            kolom_tambahan_teks = [
                'SEASONALITY', 'SUBSEGMENT', 'PRICE_SEGMENT', 'KELP_ITEM', 
                'SIZE_ITEM', 'FUNCTION_ITEM', 'BARCODE', 'FORMAT_PRODUCT', 
                'SUBFORMAT', 'OWNERSHIP', 'LEVEL_DATA'
            ]
            for col in kolom_tambahan_teks:
                if col not in df_gabungan.columns:
                    df_gabungan[col] = ''
                else:
                    # Pastikan jadi teks & ganti NaN jadi string kosong
                    df_gabungan[col] = df_gabungan[col].astype(str).replace('nan', '')

            # B. Kolom yang bersifat ANGKA (Metrik) -> Diubah jadi 0
            kolom_metrik = [
                'Volume', 'VolumeML', 'VolumePcs', 'MillRp', 'NDSelling', 'NDInStock', 
                'NDOutofStock', 'NDTotalStock', 'NPD', 'WD', 'ForwardStockML', 
                'ForwardStockPcs', 'TotalStockML', 'TotalStockPcs', 'PurchaseVolumneML', 
                'PurchaseVolumePcs', 'NumItem', 'JlhToko', 'SkuCarried'
            ]
            
            for col in kolom_metrik:
                if col not in df_gabungan.columns:
                    # Jika kolom belum ada (seperti NumItem dll), langsung isi dengan 0
                    df_gabungan[col] = 0.0
                else:
                    # Jika kolom sudah ada, paksa ubah jadi angka. 
                    # errors='coerce' akan mengubah teks aneh / string kosong '' menjadi NaN.
                    # fillna(0) akan mengubah NaN tersebut menjadi 0.
                    df_gabungan[col] = pd.to_numeric(df_gabungan[col], errors='coerce').fillna(0)

            # ========================================================
            # 6. SIMPAN KE PARQUET (DIPECAH BERDASARKAN TAHUN-BULAN)
            # ========================================================
            print("\n💾 Menyimpan ke format Parquet per Tahun-Bulan...")
            
            # Kelompokkan data berdasarkan kombinasi Tahun dan Bulan
            grup_waktu = df_gabungan.groupby(['Tahun', 'Bulan'])
            
            jumlah_file = 0
            for (tahun, bulan), df_grup in grup_waktu:
                # Lewati data yang tahun/bulannya tidak terbaca
                if tahun == 'Unknown' or bulan == 'Unknown':
                    nama_file_parquet = 'Unidentified_Periods_candy.parquet'
                else:
                    # Format nama: YYYYMM_candy.parquet (Contoh: 202506_candy.parquet)
                    nama_file_parquet = f"{tahun}{bulan}_candy.parquet"
                
                path_simpan = os.path.join(folder_excel, nama_file_parquet)
                
                # Simpan partisi ke parquet
                df_grup.to_parquet(path_simpan, index=False)
                print(f"   -> Tersimpan: {nama_file_parquet} ({len(df_grup)} baris)")
                jumlah_file += 1
                
            print("\n=======================================================")
            print(f"🎉 SELESAI! Data berhasil dipecah dan disimpan menjadi {jumlah_file} file Parquet.")
            print(f"📁 Lokasi: {folder_excel}")
            print("=======================================================")

except Exception as e:
    print(f"\n❌ Terjadi kesalahan sistem: {e}")

⏳ Memulai proses penggabungan Candy New (Konsep Parquet Bulanan)...

✅ Berhasil memuat 32 aturan perubahan nama kolom.
✅ Berhasil memuat 162 aturan perubahan nama market.
📄 Ditemukan 2 file Excel. Mulai memproses...

   -> jan26.xlsx: Dibaca 550405 baris | Data bersih: 550405 baris.
   -> jun25.xlsx: Dibaca 573686 baris | Data bersih: 573686 baris.

🔗 Menggabungkan semua data antar file...
🔄 Menerapkan Mapping Market...
🕵️ Memeriksa Duplikat Data berdasarkan Dimensi Kunci...
⚠️ DITEMUKAN 85796 BARIS DUPLIKAT! (Dihapus)
📅 Mengekstrak Tahun dan Bulan...
➕ Menambahkan kolom kosong dan merapikan tipe data...

💾 Menyimpan ke format Parquet per Tahun-Bulan...
   -> Tersimpan: 202506_candy.parquet (563019 baris)
   -> Tersimpan: 202601_candy.parquet (475276 baris)

🎉 SELESAI! Data berhasil dipecah dan disimpan menjadi 2 file Parquet.
📁 Lokasi: D:/01. Project/Top 10/Candy New


In [29]:
df_candy = pd.read_csv('D:/01. Project/Top 10/Candy New/candy_new.csv')

In [33]:
unique_market = df_candy[['Market', 'Universe']].drop_duplicates().to_csv('market_candy_new.csv')